# Exercise 03 — Gated Linear Units (GLU & friends)

## Your task

Build the GLU family as `nn.Module` subclasses. A *gate* is a second learned branch that
multiplies the first one element-wise, deciding how much of each feature passes through:

$$\text{content} = W_c^{\top}[1; x]$$
$$\text{gate} = \sigma\bigl(W_g^{\top}[1; x]\bigr) \in (0, 1)$$
$$y = \text{content} \odot \text{gate}$$

As in the other exercises you implement **only `forward`**. Every piece is an op the engine
already differentiates (`@`, `*`, `exp`, ...), so once the forward is built the engine
produces the backward pass for you — no derivative to write. The gradient check at the
bottom (the same harness as Exercise 02) proves it.

## Relation to the other exercises

- [Exercise 01](q01_activations.ipynb): there you *added* `sigmoid` and `swish` as
  primitive ops with hand-written `_backward`. Here the `sigmoid` gate is instead
  **composed** from engine ops (no `_backward` needed). The `SwiGLU` class goes the other
  way and **reuses the `swish` you implemented in Ex. 01** as its gate — closing the loop
  between the two notebooks.
- [Exercise 02](q02_rewrite_the_stars.ipynb): the `BilinearUnit` below (`left * right`, two
  unrestricted branches) is exactly `StarLinear` with the identity activation. A GLU is the
  same "star" shape, but with one branch squashed into $(0, 1)$ so it acts as a gate. We
  even reuse that exercise's `gradient_check`.

## The maths of the backward (for understanding — you do not code it)

With $y = c \odot g$ the engine applies the product rule it already knows:
$\partial L/\partial c = \partial L/\partial y \odot g$ and
$\partial L/\partial g = \partial L/\partial y \odot c$; the sigmoid/tanh/... factors come
from those ops' own backward rules, chained automatically.

## Required reading

[1] Dauphin, Y. N., Fan, A., Auli, M., & Grangier, D. (2017). *Language Modeling with
Gated Convolutional Networks.* ICML 2017, PMLR 70, pp. 933-941.

[2] Shazeer, N. (2020). *GLU Variants Improve Transformer.* (ReGLU/GEGLU/SwiGLU.)

## Conventions

Column-oriented, like the rest of the library: `x` is `(in_features, batch)`, each
`nn.Linear` computes `Wᵀ @ [1; x]`, and `*` is element-wise.

## How to work

1. Run the setup cell, then bring your Exercise 01 `swish` over (next cell).
2. Fill in the `forward` methods (remove each `raise NotImplementedError`).
3. Run the **grading** cell and the **gate statistics** cell.

> **Tip:** the gate activations you need are given below as plain functions
> `Tensor -> Tensor`. Your forwards only have to combine two `nn.Linear` branches with `*`
> — look at `StarLinear` in Exercise 02 for the shape.

In [ ]:
# Run me first: make ``bert_cpu`` importable whether Jupyter was started in the
# project root or inside ``exercises/``, exactly like the scripts in this folder do.
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "bert_cpu").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from typing import Callable

import numpy as np

from bert_cpu import engine as cpu
from bert_cpu import nn
from bert_cpu.engine import Tensor
from exercises.grading import gradient_check, report

print("ready — engine imported from", ROOT)

## Bring your Exercise 01 `swish` over

`SwiGLU` (the last unit below) gates with **Swish**, which the engine does not provide —
you hand-wrote it, forward *and* `_backward`, in [Exercise 01](q01_activations.ipynb).
Notebooks cannot import each other, so paste that implementation here.

If the SwiGLU gradient check passes at the end, it cross-validates *both* your forward here
and your Exercise 01 `swish` backward at once.

In [ ]:
class ExTensor(Tensor):
    """Your Exercise 01 tensor — only ``swish`` is needed here."""

    def swish(self) -> Tensor:
        """Swish / SiLU, x * sigmoid(x). Paste your Exercise 01 implementation."""
        # TODO: 
        raise NotImplementedError("bring ExTensor.swish over from Exercise 01")

## GIVEN — gate activations

A composition of existing engine ops. These need no hand-written backward: the engine
differentiates `exp`, `*` and `+` for us. (Contrast Exercise 01, where you add them as
fused primitive ops.)

In [ ]:
Activation = Callable[[cpu.Tensor], cpu.Tensor]


def sigmoid(t: cpu.Tensor) -> cpu.Tensor:
    """Logistic sigmoid s(t) = 1 / (1 + e^{-t}), built from engine ops only."""
    return 1.0 / (1.0 + (-t).exp())

## Exercise 1 — the Gated Linear Unit

GLU: a linear *content* branch gated by a sigmoid branch,

`y = (W_cᵀ @ [1;x]) * σ(W_gᵀ @ [1;x])`

The content branch stays linear (a direct, un-squashed path for gradients, the paper's main
argument), while the sigmoid gate in `(0, 1)` attenuates or preserves each content feature.

In [ ]:
class GatedLinearUnit(nn.Module):
    """GLU: a linear content branch gated by a sigmoid branch."""

    def __init__(self, in_features: int, out_features: int) -> None:
        self.content = nn.Linear(in_features, out_features)
        self.gate = nn.Linear(in_features, out_features)

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # TODO: 
        raise NotImplementedError("implement GatedLinearUnit.forward")

## Exercise 2 — related gated/bilinear units

`BilinearUnit` — pure bilinear unit, `y = (W_lᵀ @ [1;x]) * (W_rᵀ @ [1;x])`. Neither branch
is squashed, so this is *not* a gate — it is exactly the `StarLinear` of Exercise 02 with
the identity activation. Comparing it with the GLU shows what the sigmoid actually adds: a
bounded, interpretable gate.

`GatedTanhUnit` (GTU) — a tanh *value* branch gated by a sigmoid branch,
`y = tanh(W_vᵀ @ [1;x]) * σ(W_gᵀ @ [1;x])`. The GLU paper compares against this and argues
GLU's linear content branch propagates gradients more directly than GTU's tanh-squashed
one.

In [ ]:
class BilinearUnit(nn.Module):
    """Pure bilinear unit: ``y = left * right`` (= StarLinear with the identity)."""

    def __init__(self, in_features: int, out_features: int) -> None:
        self.left = nn.Linear(in_features, out_features)
        self.right = nn.Linear(in_features, out_features)

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # TODO: 
        raise NotImplementedError("implement BilinearUnit.forward")

In [ ]:
class GatedTanhUnit(nn.Module):
    """GTU: a tanh value branch gated by a sigmoid branch."""

    def __init__(self, in_features: int, out_features: int) -> None:
        self.value = nn.Linear(in_features, out_features)
        self.gate = nn.Linear(in_features, out_features)

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # TODO: 
        raise NotImplementedError("implement GatedTanhUnit.forward")

## Exercise 3 — modern GLU variants (ReGLU / GEGLU / SwiGLU)

`ActivatedGatedUnit` is the generalised gated unit, `y = content * gate_activation(gate_pre)`.
Choosing `gate_activation` recovers each named variant. ReGLU and GEGLU gate with
activations the engine already provides as methods:

| variant | gate activation |
|---|---|
| GLU | `sigmoid` (see `GatedLinearUnit` above) |
| ReGLU | `Tensor.relu` (`build_reglu`) |
| GEGLU | `Tensor.gelu` (`build_geglu`) |

SwiGLU gates with `swish`, which is *not* a built-in method — so it gets its own class
below, reusing the `swish` you brought over from Exercise 01. `nn.Linear` returns a plain
`Tensor`, so call your method on it as a function — a method is just a function of a
tensor: `ExTensor.swish(self.gate(x))`.

In [ ]:
class ActivatedGatedUnit(nn.Module):
    """Generalised gated unit: ``y = content * gate_activation(gate_pre)``."""

    def __init__(self, in_features: int, out_features: int, gate_activation: Activation) -> None:
        self.gate_activation = gate_activation
        self.content = nn.Linear(in_features, out_features)
        self.gate = nn.Linear(in_features, out_features)

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # TODO: 
        raise NotImplementedError("implement ActivatedGatedUnit.forward")

In [ ]:
class SwiGLU(nn.Module):
    """SwiGLU: content gated by **Swish** — ``y = content * swish(gate_pre)``."""

    def __init__(self, in_features: int, out_features: int) -> None:
        self.content = nn.Linear(in_features, out_features)
        self.gate = nn.Linear(in_features, out_features)

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # TODO: 
        raise NotImplementedError("implement SwiGLU.forward")

## GIVEN — the builders and the grading harness

You do not edit the cells below.

In [ ]:
def build_reglu(in_features: int, out_features: int) -> ActivatedGatedUnit:
    return ActivatedGatedUnit(in_features, out_features, lambda t: t.relu())


def build_geglu(in_features: int, out_features: int) -> ActivatedGatedUnit:
    return ActivatedGatedUnit(in_features, out_features, lambda t: t.gelu())


def gate_statistics(gate: cpu.Tensor) -> dict:
    """Summarise a sigmoid gate: how open/closed are its values?"""
    g = gate.data
    return {
        "mean": float(g.mean()),
        "std": float(g.std()),
        "min": float(g.min()),
        "max": float(g.max()),
        "frac<0.1 (closed)": float(np.mean(g < 0.1)),
        "frac>0.9 (open)": float(np.mean(g > 0.9)),
    }

In [ ]:
def grade() -> None:
    """Gradient-check every unit (smooth gates -> finite differences agree)."""
    cpu.set_seed(0)
    n_in, n_out, batch = 3, 2, 4
    x = cpu.Tensor(np.random.randn(n_in, batch))
    target = cpu.Tensor(np.random.randn(n_out, batch))

    cases = [
        ("GatedLinearUnit (GLU)", GatedLinearUnit(n_in, n_out)),
        ("BilinearUnit (= StarLinear)", BilinearUnit(n_in, n_out)),
        ("GatedTanhUnit (GTU)", GatedTanhUnit(n_in, n_out)),
        ("GEGLU", build_geglu(n_in, n_out)),
        ("SwiGLU (uses your Ex.01 swish)", SwiGLU(n_in, n_out)),
    ]
    print("Gradient check (analytic backward vs finite differences):\n")
    all_ok = True
    for name, layer in cases:
        print(f"  {name}:")
        try:
            worst = gradient_check(layer, x, target)
        except NotImplementedError as exc:
            # e.g. SwiGLU needs ExTensor.swish, which you wrote in Ex. 01.
            all_ok = False
            print(f"    -> SKIPPED ({exc}). Finish that piece first.\n")
            continue
        ok = worst < 1e-5
        all_ok = all_ok and ok
        print(f"    -> {'PASS' if ok else 'FAIL'} (worst {worst:.2e})\n")
    report(all_ok, "units")


grade()

## What the gate actually does

A sigmoid gate is a learned, bounded on/off switch: every value lands in `(0, 1)`, so the
statistics below tell you how much of the content branch each unit lets through.

In [ ]:
def demonstrate_gate() -> None:
    """Show the sigmoid gate of a GLU: a learned, bounded on/off switch in (0,1)."""
    cpu.set_seed(1)
    glu = GatedLinearUnit(6, 6)
    x = cpu.Tensor(np.random.randn(6, 8))
    glu(x)                                   # populate the graph
    gate = sigmoid(glu.gate(x))              # the gate tensor itself
    print("GLU gate statistics on a random batch (sigmoid -> always in (0, 1)):")
    for key, value in gate_statistics(gate).items():
        print(f"    {key:18s} {value:+.3f}")


try:
    demonstrate_gate()
except NotImplementedError as exc:
    print(f"Implement the forwards first ({exc}).")

---

**Next:** [Exercise 04 — Learnable Activations](q04_learnable_activations.ipynb). Those
*gate* or *multiply* branches; the next one **adds** activations with learned weights.